# 23. Does learning task 15 repair the damage?

Notebook 22 left one question that decides whether any of this is usable.

### Where we are

Two ways of forgetting, both measured at the moment forgetting stops:

| | forgotten task | the other seven tasks |
| --- | --- | --- |
| Published method, 100 steps | not noise, just switched off | 14.3% |
| Fixed target, 300 steps | still switched off | 21.9% |
| Fixed target, 800 steps | genuinely noise-like | 17.0% |

The other seven tasks started at 40.4%, so all three look bad. But that is not where
your study measured them. Your study measured after task 15 had been learned, and there
the same tasks had climbed back to roughly 30 to 50%.

So learning repairs a lot of the damage. Nobody has checked whether it still repairs the
damage when the fixed target is used.

### The question

If the repair works, then 17% right after forgetting is not the final answer, and the
fixed target is usable. If the repair does not work, the fixed target buys real forgetting
at a price nobody would pay.

### What this notebook runs

Five models, all starting from the same saved source model, each ending with task 15
learned exactly as the study learns it:

1. **Control**: learn task 15 with no forgetting at all. This is your study's control and
   sets the bar for everything else.
2. **Published, 100 steps**: the study's own branch, for comparison.
3. **Fixed target, 100 steps**
4. **Fixed target, 300 steps**: where the other tasks looked least damaged.
5. **Fixed target, 800 steps**: where the forgotten task genuinely reaches noise.

Each is measured three times: at the start, right after forgetting, and after task 15 is
learned.

### What we are looking for

- **Repair**: do the other seven tasks come back, and how close to the control?
- **Relapse**: does task 3 come back to life once learning resumes? Your study treats
  anything above 12% as a relapse.
- **Forward learning**: does task 15 still reach the control's accuracy?

Nothing is written into the study, and the repository stays at its pinned revision.

## 1. Load the fixed code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "65e2e7c850fd6f1852a99d9e81780c27f799c926"
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading this fixed code revision.")
if IN_COLAB:
    REPO = Path("/content/cark-paired-study")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
    if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
        raise RuntimeError("Checkout has edits. Use a fresh runtime.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a CARK checkout.")
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    implementation = ["uncle", "pyproject.toml", "requirements.txt"]
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--", *implementation], text=True).strip()
    # A later documentation-only commit may contain the same fixed implementation.
    differences = subprocess.check_output(["git", "-C", str(REPO), "diff", CODE_VERSION, actual, "--", *implementation], text=True).strip()
    if dirty or differences:
        raise RuntimeError("Use a clean checkout of CODE_VERSION, or open in Colab. Local training edits are preserved.")
for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies before running this notebook.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))
from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config, random_stream
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.study_training import restore_rng
# The progress panel only draws bars, so it cannot change a result. It
# therefore sits outside the pinned implementation, in scripts/, and this
# checkout is detached to CODE_VERSION, which predates it. Try the checkout
# first, then main, then fall back to ordinary bars rather than failing.
sys.path.insert(0, str(REPO / 'scripts'))
try:
    from dashboard import Dashboard
except ImportError:
    try:
        tools = Path.cwd() / '_tools'
        tools.mkdir(exist_ok=True)
        subprocess.run(['git', '-C', str(REPO), 'fetch', '--quiet', 'origin', 'main'], check=True)
        (tools / 'dashboard.py').write_bytes(subprocess.check_output(
            ['git', '-C', str(REPO), 'show', 'origin/main:scripts/dashboard.py']))
        sys.path.insert(0, str(tools))
        from dashboard import Dashboard
        print('Progress panel fetched from main.')
    except Exception as problem:
        print('Progress panel unavailable, using plain bars:', problem)

        class Dashboard:
            """Just enough of the panel to keep the notebook running."""

            def __init__(self, *args, **kwargs): pass
            def __enter__(self): return self
            def __exit__(self, *args): return False
            def note(self, text): print(text, flush=True)

            def bar(self, total, description, unit='step'):
                from uncle.telemetry import progress_bar
                return progress_bar(total, description)

import torch
print("Fixed implementation revision:", CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)

## 2. Settings

`BUDGETS` is the list of forgetting step counts to try before learning task 15. Everything
else matches the saved study.

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"

ORDER = "order_01"
SEED = 0                 # 0, 1, or 2
FORGET_TASK = "3"        # "3" or "14"

# How many forgetting steps to try before learning task 15. 100 is your
# study's setting. 300 is where the other tasks looked least damaged. 800 is
# where the fixed target first reaches genuinely noise-like weights.
BUDGETS = [100, 300, 800]

# Your study calls the task forgotten if it stays at or below this score, and
# counts anything above it after later learning as a relapse.
RELAPSE_LIMIT = 12.0

SAVE_TO_R2 = True
RELEASE_GPU_WHEN_DONE = True

if FORGET_TASK not in ("3", "14") or SEED not in (0, 1, 2):
    raise ValueError("The study only completed seeds 0-2 forgetting task 3 or 14.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Forgetting budgets to try:", BUDGETS)
print("Runs:", 2 + len(BUDGETS), "| each one learns task 15, which is the slow part")

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID`, and `R2_SECRET_ACCESS_KEY` through Colab Secrets
or environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Load the study and the source model

Task 15's images are loaded too, because this notebook actually learns it. The two
protected lists are copied from the study runner: forgetting protects every task except
its own target, and learning protects every task including the forgotten one.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
path = store.download(PREFIX + "/study.json", CACHE / "cache")
plan = json.loads(path.read_text())

if plan["code_version"] != CODE_VERSION:
    raise ValueError("This study requires a different fixed code revision.")
from uncle.learning_diagnostic import file_hash
if file_hash(REPO / "uncle/task_partition.json") != plan["partition_sha256"]:
    raise ValueError("Class partition differs from the saved study.")

storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = tuple(plan["orders"][ORDER])
NEW_TASK = plan["new_tasks"][0]
CLASSES = plan["config"]["classes_per_task"]
EVAL_BATCH = plan["config"]["eval_batch_size"]

# Task 15 is learned here, so its images are needed too.
tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=ORDER_TASKS + (NEW_TASK,), download=True)

source_job = next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{SEED}")
branch_job = next(j for j in ALL_JOBS
                  if j["id"] == f"branches/{ORDER}/seed_{SEED}/forget_{FORGET_TASK}/learn_{NEW_TASK}")
state = storage.load(source_job)
if state is None or state["report"]["status"] != "complete":
    raise RuntimeError(f"Source model for seed {SEED} is not finished in R2.")

REQUESTS = tuple(("learn", t) for t in ORDER_TASKS)
REQUESTS += tuple(tuple(op) for op in branch_job["operations"])

# Forgetting protects every learned task except its own target. Learning
# protects every learned task including the forgotten one, which is what keeps
# a forgotten task down while something new is learned. Both lines match the
# study runner exactly.
PROTECT_WHILE_FORGETTING = [t for t in state["seen"] if t != FORGET_TASK]
PROTECT_WHILE_LEARNING = list(state["seen"])
print("Source:", source_job["id"], "| learning task", NEW_TASK, "after each variant")
print("Protected while forgetting:", len(PROTECT_WHILE_FORGETTING),
      "| while learning:", len(PROTECT_WHILE_LEARNING))
print("Epochs for task 15:", plan["config"]["epochs"])

## 5. The measurement

The same measurements as notebooks 21 and 22, plus task 15, taken at three moments in each
run.

In [ ]:
from torch.func import functional_call
from torch.utils.data import DataLoader

def inspect(hypernet, target, trainer, task, full=False):
    """Accuracy, and optionally how the model uses its input."""
    weights = hypernet.weights_for(task)
    buffers = {name: value.clone() for name, value in trainer.task_buffers[task].items()}
    loader = DataLoader(tasks[task]["test"], batch_size=EVAL_BATCH)
    picked, correct, total = [], 0, 0
    lowest = highest = None
    with torch.no_grad():
        for images, labels in loader:
            scores = functional_call(target, (weights, buffers),
                                     (images.to(trainer.device),), strict=True)
            choice = scores.argmax(1)
            picked.append(choice.cpu())
            correct += (choice == labels.to(trainer.device)).sum().item()
            total += labels.numel()
            if full:
                low, high = scores.min(dim=0).values, scores.max(dim=0).values
                lowest = low if lowest is None else torch.minimum(lowest, low)
                highest = high if highest is None else torch.maximum(highest, high)
    result = {"accuracy": 100.0 * correct / total}
    if full:
        counts = torch.bincount(torch.cat(picked), minlength=CLASSES).tolist()
        result["distinct_classes"] = sum(1 for c in counts if c > 0)
        result["input_sensitivity"] = (highest - lowest).max().item()
    return result

def measure_all(hypernet, target, trainer, stage):
    """One full reading: the forgotten task, the seven others, and task 15."""
    was_training = hypernet.training
    hypernet.eval(); target.eval()
    forgotten = inspect(hypernet, target, trainer, FORGET_TASK, full=True)
    others = {t: inspect(hypernet, target, trainer, t)["accuracy"]
              for t in ORDER_TASKS if t != FORGET_TASK}
    row = {
        "stage": stage,
        "forgotten_accuracy": forgotten["accuracy"],
        "distinct_classes": forgotten["distinct_classes"],
        "input_sensitivity": forgotten["input_sensitivity"],
        "weight_size": hypernet.raw_for(FORGET_TASK).norm().item(),
        "other_tasks": others,
        "others_mean": sum(others.values()) / len(others),
        # Task 15 has no code until it is learned, so this is None before then.
        "new_task_accuracy": (inspect(hypernet, target, trainer, NEW_TASK)["accuracy"]
                              if NEW_TASK in hypernet.task_codes else None),
    }
    new = "-" if row["new_task_accuracy"] is None else f"{row['new_task_accuracy']:.1f}%"
    print(f"    {stage:<28} forgotten {row['forgotten_accuracy']:5.1f}% | "
          f"others {row['others_mean']:5.1f}% | task {NEW_TASK} {new:>6} | "
          f"sensitivity {row['input_sensitivity']:.3e}")
    if was_training:
        hypernet.train()
    return row

def build(steps=None):
    """The model as the study runner builds it, loaded with the saved weights."""
    # run_request reads the step count from the config, so it is set here.
    config = Config(**{**plan["config"], "seed": SEED, "device": DEVICE,
                       "requests": REQUESTS,
                       "burn_in": steps or plan["config"]["burn_in"]})
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=True)
    checkpoint.restore(state, hypernet=hypernet, uncle=trainer)
    restore_rng(state["all_rng"])
    return config, target, hypernet, trainer

## 6. The forgetting loop

Unchanged from notebook 22, where it was checked against the study code and matched it
exactly.

In [ ]:
import uncle.study_training as study_training
from uncle.paired_diagnostic import generated_noise_loss

def fixed_target_noise_loss(hypernet, task, noise, samples):
    """The same loss, aimed at one target drawn once instead of many fresh ones.

    The study's own loss draws `samples` new random targets every step and
    averages the distance to them. The paper proves in its Theorem 1 that this
    equals the squared size of the weights plus a constant, so it pulls the
    weights toward zero rather than toward noise.

    This version draws a single target the first time it is called and reuses
    it for every step, so the closest point is that target itself, which is
    real noise. Everything else matches the study's function exactly.
    """
    weights = hypernet.weights_for(task)
    generated = torch.cat([value.reshape(-1) for value in weights.values()])
    if task not in fixed_target_noise_loss.targets:
        fixed_target_noise_loss.targets[task] = torch.randn(
            generated.shape, generator=noise,
            device=generated.device, dtype=generated.dtype)
    return (generated - fixed_target_noise_loss.targets[task]).square().sum()

fixed_target_noise_loss.targets = {}

def study_forget(trainer, task, protected, *, fixed_target, steps, on_step=None):
    """One forgetting request through the study's own request runner.

    Nothing is reimplemented here. The only difference between the two
    variants is which noise loss `run_request` sees, swapped in for the
    duration of the call and put back afterwards. The step count comes from
    the config the caller built, because run_request reads config.burn_in.
    """
    if trainer.config.burn_in != steps:
        raise ValueError("Build the config with burn_in equal to the step count.")
    fixed_target_noise_loss.targets.clear()
    original = study_training.generated_noise_loss
    losses = []

    def note(row):
        if "loss" in row:
            losses.append(row["loss"])
        if on_step is not None:
            on_step(row.get("step", 0))

    study_training.generated_noise_loss = (
        fixed_target_noise_loss if fixed_target else original)
    try:
        # forget_interval=steps avoids copying optimiser state for a resume
        # that will not happen. evaluation_interval is out of range because
        # the caller takes its own richer measurements.
        study_training.run_request(trainer, "forget", task, protected, None, note,
                                   lambda active: None, forget_interval=steps,
                                   evaluation_interval=10 ** 9)
    finally:
        study_training.generated_noise_loss = original
    return losses

## 7. Run every variant

Each one rebuilds from the saved weights, optionally forgets, then learns task 15 the same
way. Learning is the slow part, so expect this to take a while.

In [ ]:
import gc

runs = {}

def variant(label, *, forget_steps=None, fixed_target=None):
    """Optionally forget, then always learn task 15, measuring at each stage."""
    config, target, hypernet, trainer = build(steps=forget_steps)
    stages = [measure_all(hypernet, target, trainer, "start")]

    if forget_steps:
        study_forget(trainer, FORGET_TASK, PROTECT_WHILE_FORGETTING,
                     fixed_target=bool(fixed_target), steps=forget_steps)
        stages.append(measure_all(hypernet, target, trainer,
                                  f"after forgetting ({forget_steps})"))

    # Learning is identical in every variant: same task, same protected list,
    # same epochs. Only what happened before it differs.
    #
    # trainer.learn is used rather than the study's run_request because the
    # two were checked against each other and produce bit-identical weights,
    # and this one skips run_request's per-step input hashing. Forgetting is
    # the opposite case: there the two paths genuinely differ, which is why
    # study_forget above goes through run_request.
    trainer.learn(NEW_TASK, PROTECT_WHILE_LEARNING)
    stages.append(measure_all(hypernet, target, trainer, f"after learning task {NEW_TASK}"))

    runs[label] = {"stages": stages, "forget_steps": forget_steps or 0,
                   "fixed_target": fixed_target}
    del trainer, hypernet, target
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

plan_of_runs = [("control: no forgetting", None, None),
                (f"published: fresh {BUDGETS[0]}", BUDGETS[0], False)]
plan_of_runs += [(f"fixed {budget}", budget, True) for budget in BUDGETS]

with Dashboard(f"Learn after unlearn: seed {SEED}, forget task {FORGET_TASK}") as board:
    overall = board.bar(len(plan_of_runs), "Variants", unit="run")
    for label, steps, fixed in plan_of_runs:
        board.note(label)
        print(f"\n=== {label} ===")
        variant(label, forget_steps=steps, fixed_target=fixed)
        overall.update(1)
    overall.close()
    board.note(f"measured {len(runs)} variants")

print(f"\nMeasured {len(runs)} variants.")

## 8. Compare

The first table asks whether learning repaired the other seven tasks. The second asks
whether the forgotten task stayed forgotten and whether task 15 still learned.

In [ ]:
control = runs["control: no forgetting"]["stages"][-1]
chance = 100.0 / CLASSES

def summarise(label):
    run = runs[label]
    start, end = run["stages"][0], run["stages"][-1]
    after_forget = run["stages"][1] if len(run["stages"]) > 2 else None
    return {
        "run": label,
        "steps": run["forget_steps"],
        "others_start": start["others_mean"],
        "others_after_forget": after_forget["others_mean"] if after_forget else None,
        "others_final": end["others_mean"],
        "others_vs_control": end["others_mean"] - control["others_mean"],
        "forgotten_after_forget": after_forget["forgotten_accuracy"] if after_forget else None,
        "forgotten_final": end["forgotten_accuracy"],
        "relapsed": end["forgotten_accuracy"] > RELAPSE_LIMIT,
        "new_task": end["new_task_accuracy"],
        "new_task_vs_control": end["new_task_accuracy"] - control["new_task_accuracy"],
        "sensitivity_after_forget": after_forget["input_sensitivity"] if after_forget else None,
        "sensitivity_final": end["input_sensitivity"],
    }

table = [summarise(label) for label in runs]

print("DID LEARNING REPAIR THE OTHER SEVEN TASKS?")
print(f"{'run':<24} {'steps':>6} {'at start':>9} {'after forget':>13} {'after learning':>15} {'vs control':>11}")
print("-" * 83)
for r in table:
    mid = "-" if r["others_after_forget"] is None else f"{r['others_after_forget']:.1f}%"
    print(f"{r['run']:<24} {r['steps']:>6} {r['others_start']:8.1f}% {mid:>13}"
          f" {r['others_final']:14.1f}% {r['others_vs_control']:+10.1f}")

print()
print("DID THE FORGOTTEN TASK STAY FORGOTTEN, AND DID TASK 15 STILL LEARN?")
print(f"{'run':<24} {'forgotten':>10} {'relapsed':>9} {'task ' + NEW_TASK:>8} {'vs control':>11} {'sensitivity':>13}")
print("-" * 80)
for r in table:
    print(f"{r['run']:<24} {r['forgotten_final']:9.1f}% {str(r['relapsed']):>9}"
          f" {r['new_task']:7.1f}% {r['new_task_vs_control']:+10.1f} {r['sensitivity_final']:13.3e}")

print()
print("VERDICT")
print(f"  The control ends with the other tasks at {control['others_mean']:.1f}% and"
      f" task {NEW_TASK} at {control['new_task_accuracy']:.1f}%.")
print("  Everything else is judged against those two numbers.")
print()
for r in table:
    if r["steps"] == 0:
        continue
    repaired = r["others_after_forget"] is not None and r["others_final"] > r["others_after_forget"] + 1.0
    close = r["others_vs_control"] > -5.0
    learned = r["new_task_vs_control"] > -5.0
    noise_like = (r["sensitivity_after_forget"] or 0) > 1.0
    verdict = []
    verdict.append("repaired" if repaired else "no repair")
    verdict.append("retention near control" if close else "retention well below control")
    verdict.append("task 15 fine" if learned else "task 15 hurt")
    verdict.append("reached noise" if noise_like else "never reached noise")
    verdict.append("RELAPSED" if r["relapsed"] else "stayed forgotten")
    print(f"  {r['run']:<24} " + "; ".join(verdict))

print()
usable = [r for r in table if r["steps"] and not r["relapsed"]
          and r["others_vs_control"] > -5.0 and r["new_task_vs_control"] > -5.0
          and (r["sensitivity_after_forget"] or 0) > 1.0]
if usable:
    best = max(usable, key=lambda r: r["others_final"])
    print(f"  USABLE: {best['run']} reaches real noise and still ends within 5 points")
    print("  of the control on both the other tasks and task 15.")
else:
    print("  NONE of these settings both reaches real noise and keeps the other tasks")
    print("  near the control. Forgetting properly and keeping the rest appear to")
    print("  pull against each other in this method.")

## 9. Chart

Left: the seven protected tasks, before and after learning, against the control. Middle:
task 15. Right: the forgotten task against the relapse limit.

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime, timezone

EXPORT = CACHE / "diagnostics" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPORT.mkdir(parents=True, exist_ok=True)

labels = list(runs)
colours = plt.cm.viridis([i / max(1, len(labels) - 1) for i in range(len(labels))])
figure, axes = plt.subplots(1, 3, figsize=(16, 4.8))

positions = range(len(labels))
axes[0].bar([p - 0.2 for p in positions],
            [runs[l]["stages"][1]["others_mean"] if len(runs[l]["stages"]) > 2
             else runs[l]["stages"][0]["others_mean"] for l in labels],
            0.4, label="after forgetting", color="#d08c60")
axes[0].bar([p + 0.2 for p in positions],
            [runs[l]["stages"][-1]["others_mean"] for l in labels],
            0.4, label="after learning task " + NEW_TASK, color="#245d99")
axes[0].axhline(control["others_mean"], color="black", linestyle="--", linewidth=1,
                label="control")
axes[0].axhline(chance, color="grey", linestyle=":", linewidth=1)
axes[0].set_title("The seven protected tasks"); axes[0].set_ylabel("accuracy (%)")

axes[1].bar(positions, [runs[l]["stages"][-1]["new_task_accuracy"] for l in labels],
            0.6, color="#4c956c")
axes[1].axhline(control["new_task_accuracy"], color="black", linestyle="--", linewidth=1,
                label="control")
axes[1].set_title(f"Task {NEW_TASK} after learning"); axes[1].set_ylabel("accuracy (%)")

axes[2].bar(positions, [runs[l]["stages"][-1]["forgotten_accuracy"] for l in labels],
            0.6, color="#b8322a")
axes[2].axhline(RELAPSE_LIMIT, color="black", linestyle="--", linewidth=1,
                label=f"relapse limit {RELAPSE_LIMIT:.0f}%")
axes[2].axhline(chance, color="grey", linestyle=":", linewidth=1)
axes[2].set_title(f"Forgotten task {FORGET_TASK} at the end"); axes[2].set_ylabel("accuracy (%)")

for axis in axes:
    axis.set_xticks(list(positions))
    axis.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    axis.legend(fontsize=8)

figure.suptitle(f"{STUDY_ID} | {ORDER} seed {SEED} | forgetting, then learning task {NEW_TASK}")
figure.tight_layout()
figure.savefig(EXPORT / "learn_after_unlearn.png", dpi=180)
figure.savefig(EXPORT / "learn_after_unlearn.svg")
plt.show()
print("Figure written to", EXPORT)

## 10. Save

Writes a summary, a comparison table and a per-stage table next to the figure, then copies
them to R2 under a diagnostics folder separate from the study's own files.

In [ ]:
import csv

summary = {
    "study_id": STUDY_ID, "code_version": CODE_VERSION, "order": ORDER, "seed": SEED,
    "forget_task": FORGET_TASK, "new_task": NEW_TASK, "device": DEVICE,
    "budgets": BUDGETS, "relapse_limit": RELAPSE_LIMIT,
    "classes_per_task": CLASSES, "epochs": plan["config"]["epochs"],
    "gamma": plan["config"]["gamma"], "beta": plan["config"]["beta"],
    "control": control, "comparison": table,
    "runs": {label: run for label, run in runs.items()},
    "reading": ("Measured at three moments: before forgetting, right after forgetting, "
                "and after task 15 is learned. The study only ever reported the last of "
                "these, which hides how much damage the middle stage does."),
}
(EXPORT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

with (EXPORT / "comparison.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(table[0]))
    writer.writeheader()
    writer.writerows(table)

with (EXPORT / "stages.csv").open("w", newline="", encoding="utf-8") as stream:
    columns = ["run", "forget_steps", "stage", "forgotten_accuracy", "others_mean",
               "new_task_accuracy", "input_sensitivity", "weight_size",
               "distinct_classes"] + [f"task_{t}" for t in ORDER_TASKS if t != FORGET_TASK]
    writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    for label, run in runs.items():
        for stage in run["stages"]:
            line = {"run": label, "forget_steps": run["forget_steps"]}
            line.update({k: v for k, v in stage.items() if k != "other_tasks"})
            line.update({f"task_{t}": v for t, v in stage["other_tasks"].items()})
            writer.writerow(line)

print("Saved locally:")
for item in sorted(EXPORT.iterdir()):
    print("  ", item)

if SAVE_TO_R2:
    destination = PREFIX + "/diagnostics/" + EXPORT.name
    with Dashboard("Saving results") as board:
        files = sorted(EXPORT.iterdir())
        sending = board.bar(len(files), "Upload diagnostics")
        for item in files:
            sending.set_postfix_str(item.name)
            store.upload(item, destination + "/" + item.name)
            sending.update(1)
        sending.close()
    print("Uploaded to", destination)
    SAVED_LOCATION = destination
else:
    SAVED_LOCATION = str(EXPORT)
    print()
    print("SAVE_TO_R2 is off, so the only copy is inside this runtime.")

EXPECTED_FILES = ["comparison.csv", "learn_after_unlearn.png",
                  "learn_after_unlearn.svg", "stages.csv", "summary.json"]
SAVE_VERIFIED = sorted(item.name for item in EXPORT.iterdir()) == EXPECTED_FILES
print()
print("All expected files present:", SAVE_VERIFIED)

## 11. Hand the GPU back

Releases the runtime once the work is saved. It refuses if any expected file is missing.

In [ ]:
if not SAVE_VERIFIED:
    print("Not releasing the GPU: some output files are missing.")
    print("Rerun section 10 before closing this runtime, or the results are lost.")
elif not RELEASE_GPU_WHEN_DONE:
    print("Results saved to:", SAVED_LOCATION)
    print("RELEASE_GPU_WHEN_DONE is off, so the runtime stays connected.")
elif not IN_COLAB:
    print("Results saved to:", SAVED_LOCATION)
    print("Not running in Colab, so there is no runtime to release.")
else:
    print("Results saved to:", SAVED_LOCATION)
    print("Share that path.")
    import gc
    for name in ("runs", "state", "tasks"):
        globals().pop(name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    from google.colab import runtime
    print("Releasing the Colab runtime now.")
    runtime.unassign()

## What to do with the output

Send me the two tables and the verdict from section 8.

**If a setting both reaches noise and keeps the other tasks near the control**, you have a
working method and a clear story: the published version fails, this one line fixes it, and
here is the budget it needs.

**If none of them does**, you have something harder but more interesting: in this method,
truly erasing one task and keeping the rest pull against each other. The published version
hides that tension by never erasing anything in the first place.

Either way, check the relapse column. If the forgotten task climbs back above 12% once
learning resumes, then the forgetting did not hold, and that matters more than the
retention numbers.

**Still not tested.** Whether the forgotten knowledge could be recovered by someone who
tried. A model scoring 10% may still carry the information. That is the separate study in
notebooks 16 and 17, and it is the only thing that would support a privacy claim.

**Scope.** One seed, one forgotten task, one order, beta 0.1 against the paper's 0.01 for
Tiny ImageNet. Repeat across seeds before writing any of it up.